# Construcción del Modelo de Referencia (Baseline) Estadístico

El presente cuaderno establece un modelo de referencia (baseline) robusto basado en características estadísticas temporales, fundamentando la base comparativa para la validación de arquitecturas de aprendizaje profundo posteriores (como el Autoencoder).

Los objetivos planteados son:

- Ingestar los registros telemétricos de aceleración.
- Segmentar las series temporales en secuencias de 1 segundo (con muestreo a 100 Hz) integrando un solapamiento del 50%.
- Extraer descriptores estadísticos en el dominio del tiempo.
- Calibrar un umbral de decisión utilizando el comportamiento del sistema en régimen nominal.
- Evaluar el poder discriminativo de dicho umbral empleando un conjunto de evaluación heterogéneo.

Este enfoque analítico constituye la línea base que cualquier aproximación mediante redes neuronales deberá superar justificando así su mayor complejidad computacional.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

# Implementación independiente para garantizar la reproducibilidad y el desacoplamiento respecto a módulos externos.
TFM_COLUMNS = ["seq", "timestamp_ms", "acc_x_g", "acc_y_g", "acc_z_g"]
MLOPS_COLUMNS = ["timestamp", "accel_x", "accel_y", "accel_z"]

def read_vibration_csv(path):
    df = pd.read_csv(path)
    columns = set(df.columns)
    if set(TFM_COLUMNS).issubset(columns):
        return df[TFM_COLUMNS].copy()
    if set(MLOPS_COLUMNS).issubset(columns):
        converted = pd.DataFrame()
        converted["seq"] = np.arange(len(df), dtype=np.int64)
        converted["timestamp_ms"] = np.rint(df["timestamp"].astype(float) * 1000.0).astype(np.int64)
        converted["acc_x_g"] = df["accel_x"].astype(float)
        converted["acc_y_g"] = df["accel_y"].astype(float)
        converted["acc_z_g"] = df["accel_z"].astype(float)
        return converted
    raise ValueError(f"Columnas no reconocidas: {list(df.columns)}")

def make_windows(values, window_size=100, window_step=50):
    values = np.asarray(values, dtype=np.float32)
    return np.asarray(
        [values[start:start + window_size] for start in range(0, len(values) - window_size + 1, window_step)],
        dtype=np.float32,
    )

def window_features(window):
    window = np.asarray(window, dtype=np.float32)
    magnitude = np.linalg.norm(window, axis=1)
    centered = window - np.mean(window, axis=0, keepdims=True)
    features = {
        "mag_rms": float(np.sqrt(np.mean(magnitude ** 2))),
        "mag_mean": float(np.mean(magnitude)),
        "mag_std": float(np.std(magnitude)),
        "mag_peak": float(np.max(np.abs(magnitude))),
        "energy": float(np.mean(np.sum(centered ** 2, axis=1))),
    }
    for axis, name in enumerate(("x", "y", "z")):
        values = window[:, axis]
        features[f"{name}_mean"] = float(np.mean(values))
        features[f"{name}_std"] = float(np.std(values))
        features[f"{name}_rms"] = float(np.sqrt(np.mean(values ** 2)))
        features[f"{name}_peak"] = float(np.max(np.abs(values)))
    return features

def percentile_threshold(scores, percentile=99.0):
    return float(np.percentile(np.asarray(scores, dtype=np.float32), percentile))


## Configuración del Entorno de Ejecución

Es imperativo considerar la topología del sistema de archivos. En entornos contenerizados bajo ROCm, el directorio `../datos` se proyecta convencionalmente sobre `/workspace/datos`. Las rutas deberán ajustarse acorde al ecosistema de despliegue local.

In [ ]:
# Integración de la telemetría histórica, alojada localmente en el directorio estandarizado.
# Se contempla soporte dual para las arquitecturas de despliegue empleadas:
# - Topología aislada del subproyecto (contenedor específico).
# - Topología global heredada de la estructura general del proyecto.
candidate_roots = [
    Path("/workspace/inference_server/datos"),
    Path("/workspace/TFM/inference_server/datos"),
    Path("datos").resolve(),
]

DATA_ROOT = next((path for path in candidate_roots if (path / "raw").is_dir()), candidate_roots[-1])

NORMAL_CSV = DATA_ROOT / "raw" / "train_vibrations_normal.csv"
EVAL_CSV = DATA_ROOT / "raw" / "test_vibrations_anomaly.csv"

WINDOW_SIZE = 100
WINDOW_STEP = 50
THRESHOLD_PERCENTILE = 99.0

print(f"DATA_ROOT={DATA_ROOT}")
print(f"NORMAL_CSV={NORMAL_CSV} exists={NORMAL_CSV.exists()}")
print(f"EVAL_CSV={EVAL_CSV} exists={EVAL_CSV.exists()}")


## Ingestión y Estandarización de Registros

El sistema de carga ha sido diseñado para proveer retrocompatibilidad con las distintas revisiones del formato de los esquemas de datos:

- Esquema Consolidado (TFM): `seq, timestamp_ms, acc_x_g, acc_y_g, acc_z_g`
- Esquema de Desarrollo (MLOps): `timestamp, accel_x, accel_y, accel_z`

In [ ]:
normal_df = read_vibration_csv(NORMAL_CSV)
eval_df = read_vibration_csv(EVAL_CSV)

display(normal_df.head())
display(eval_df.head())

print(f"normal rows={len(normal_df):,}")
print(f"eval rows={len(eval_df):,}")

## Exploración Visual de la Dinámica Temporal

In [ ]:
def plot_signal(df: pd.DataFrame, title: str, max_rows: int = 2000) -> None:
    subset = df.iloc[:max_rows]
    t = subset["timestamp_ms"].to_numpy() / 1000.0
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, subset["acc_x_g"], label="x", linewidth=1)
    ax.plot(t, subset["acc_y_g"], label="y", linewidth=1)
    ax.plot(t, subset["acc_z_g"], label="z", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Aceleración (g)")
    ax.legend()
    plt.show()

plot_signal(normal_df, "Régimen Nominal")
plot_signal(eval_df, "Secuencia de Evaluación")

## Segmentación Temporal y Extracción de Características

In [ ]:
def compute_baseline_table(df: pd.DataFrame) -> pd.DataFrame:
    values = df[["acc_x_g", "acc_y_g", "acc_z_g"]].to_numpy(dtype=np.float32)
    windows = make_windows(values, window_size=WINDOW_SIZE, window_step=WINDOW_STEP)
    rows = []

    for idx, window in enumerate(windows):
        start = idx * WINDOW_STEP
        end = start + WINDOW_SIZE - 1
        features = window_features(window)
        features["vibration_rms"] = float(np.sqrt(features["energy"]))
        features["max_axis_std"] = float(max(features["x_std"], features["y_std"], features["z_std"]))
        features["mean_axis_std"] = float(np.mean([features["x_std"], features["y_std"], features["z_std"]]))
        features["xy_std_sum"] = float(features["x_std"] + features["y_std"])
        features["window_idx"] = idx
        features["start_seq"] = int(df.iloc[start]["seq"])
        features["end_seq"] = int(df.iloc[end]["seq"])
        features["start_ms"] = int(df.iloc[start]["timestamp_ms"])
        features["end_ms"] = int(df.iloc[end]["timestamp_ms"])
        rows.append(features)

    return pd.DataFrame(rows)

normal_features = compute_baseline_table(normal_df)
eval_features = compute_baseline_table(eval_df)

display(normal_features.head())
print(f"normal windows={len(normal_features):,}")
print(f"eval windows={len(eval_features):,}")


## Análisis Comparativo de las Funciones de Puntuación (Scoring)

Se procede a evaluar sistemáticamente diversas métricas heurísticas. El propósito primordial es superar la limitación inherente a la métrica RMS de magnitud absoluta, la cual adolece de un fuerte sesgo provocado por el componente estático gravitacional.

Técnicamente, el estimador más sólido para constituir nuestra métrica de referencia recae en `vibration_rms`: el valor eficaz (RMS) de la varianza dinámica espacial (tras sustraer la componente continua o valor medio en cada eje).

In [ ]:
candidate_scores = [
    "mag_rms",
    "vibration_rms",
    "energy",
    "mag_std",
    "max_axis_std",
    "mean_axis_std",
    "xy_std_sum",
    "x_std",
    "y_std",
    "z_std",
]

rows = []
for score_name in candidate_scores:
    threshold_candidate = percentile_threshold(normal_features[score_name].to_numpy(), THRESHOLD_PERCENTILE)
    normal_ratio = (normal_features[score_name] > threshold_candidate).mean()
    eval_ratio = (eval_features[score_name] > threshold_candidate).mean()
    ratio_gain = eval_ratio / normal_ratio if normal_ratio > 0 else np.inf
    rows.append({
        "score": score_name,
        "threshold": threshold_candidate,
        "normal_anomaly_ratio": normal_ratio,
        "eval_anomaly_ratio": eval_ratio,
        "eval_vs_normal_ratio": ratio_gain,
        "normal_mean": normal_features[score_name].mean(),
        "eval_mean": eval_features[score_name].mean(),
    })

comparison_df = pd.DataFrame(rows).sort_values("eval_vs_normal_ratio", ascending=False)
display(comparison_df)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(comparison_df["score"], comparison_df["eval_vs_normal_ratio"])
ax.axhline(1.0, color="red", linestyle="--", label="Sin mejora")
ax.set_title("Poder Discriminativo por Función de Puntuación")
ax.set_ylabel("Ratio de Anomalías Detectadas (Eval / Nominal)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.show()


## Definición del Estimador y Frontera de Decisión

Se adopta como métrica operativa preliminar `mag_rms`. Dada su extrema simplicidad analítica, un rendimiento excepcionalmente alto por parte de este estimador exigirá que los modelos de aprendizaje profundo subsiguientes demuestren ventajas operativas determinantes (p. ej., mayor interpretabilidad espacial o alta resiliencia frente a ruido no gaussiano).

In [ ]:
SCORE_COLUMN = "vibration_rms"

threshold = percentile_threshold(normal_features[SCORE_COLUMN].to_numpy(), THRESHOLD_PERCENTILE)

normal_features["score"] = normal_features[SCORE_COLUMN]
normal_features["is_anomaly"] = normal_features["score"] > threshold

eval_features["score"] = eval_features[SCORE_COLUMN]
eval_features["is_anomaly"] = eval_features["score"] > threshold

print(f"score={SCORE_COLUMN}")
print(f"threshold p{THRESHOLD_PERCENTILE}={threshold:.6f}")
print(f"normal anomaly ratio={normal_features['is_anomaly'].mean():.2%}")
print(f"eval anomaly ratio={eval_features['is_anomaly'].mean():.2%}")

## Caracterización Probabilística de las Puntuaciones

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_features["score"], bins=60, alpha=0.65, label="Régimen Nominal")
ax.hist(eval_features["score"], bins=60, alpha=0.65, label="Evaluación")
ax.axvline(threshold, color="red", linestyle="--", label="Umbral de Decisión")
ax.set_title("Distribución de la Función de Puntuación (Baseline)")
ax.set_xlabel(SCORE_COLUMN)
ax.set_ylabel("Frecuencia (Ventanas)")
ax.legend()
plt.show()

## Evolución Temporal del Estimador de Condición

In [ ]:
def plot_window_scores(features: pd.DataFrame, title: str) -> None:
    t = features["start_ms"].to_numpy() / 1000.0
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, features["score"], label="Puntuación", linewidth=1)
    ax.axhline(threshold, color="red", linestyle="--", label="Umbral Crítico")

    anomalous = features[features["is_anomaly"]]
    ax.scatter(
        anomalous["start_ms"] / 1000.0,
        anomalous["score"],
        color="red",
        s=18,
        label="Detección de Anomalía",
        zorder=3,
    )

    ax.set_title(title)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel(SCORE_COLUMN)
    ax.legend()
    plt.show()

plot_window_scores(normal_features, "Evolución Temporal en Régimen Nominal")
plot_window_scores(eval_features, "Evolución Temporal en Secuencia de Evaluación")

## Conclusiones Preliminares

La experimentación anterior consolida un marco referencial fundamentado en:

- La calibración analítica de la frontera de decisión empleando únicamente datos en estado de salud estructural nominal.
- La cuantificación del grado de desviación durante episodios anómalos.
- La capacidad de delimitación temporal de dichos eventos perturbadores.

Este paradigma empírico servirá como estándar de validación durante la posterior fase de entrenamiento y evaluación del modelo de inferencia latente (Autoencoder).